In [ ]:
# In[1]:

import pandas as pd

# Load files and parse timestamps as UTC datetimes
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Helper to build compact summary DataFrame for a file
def summarize(df, file_label, name_col=None, max_items=20):
    rows = df.shape[0]
    cmdb_sample = list(df['cmdb_id'].dropna().unique()[:max_items])
    if name_col and name_col in df.columns:
        name_sample = list(df[name_col].dropna().unique()[:max_items])
    else:
        name_sample = None
    return pd.DataFrame({
        'file': [file_label],
        'rows': [rows],
        'sample_cmdb_ids': [cmdb_sample],
        'sample_names': [name_sample]
    })

# Create summaries and first-5-rows for each file
metric_summary = summarize(metric, "metric.csv", name_col="kpi_name")
metric_head = metric.head(5)

trace_summary = summarize(trace, "trace.csv", name_col="trace_name")
trace_head = trace.head(5)

log_summary = summarize(log, "log.csv", name_col="log_name")
log_head = log.head(5)

error_summary = summarize(error_logs, "error_logs.csv", name_col=None)
error_head = error_logs.head(5)

# Display compact summaries and first 5 rows for each file
metric_summary, metric_head, trace_summary, trace_head, log_summary, log_head, error_summary, error_head

```
Out[1]:
```
```python
# Build a concise plain-English summary using previously loaded summaries/heads
metric_rows = int(metric_summary.at[0, 'rows'])
metric_cmdb_sample = metric_summary.at[0, 'sample_cmdb_ids'][:8]  # show up to 8 for brevity
metric_name_sample = metric_summary.at[0, 'sample_names'][:6]

trace_rows = int(trace_summary.at[0, 'rows'])
log_rows = int(log_summary.at[0, 'rows'])
log_cmdb_sample = log_summary.at[0, 'sample_cmdb_ids'][:8]
log_name_sample = log_summary.at[0, 'sample_names'][:6]

error_rows = int(error_summary.at[0, 'rows'])

first_metric_ts = metric_head.iloc[0]['timestamp'] if not metric_head.empty else "N/A"

summary = (
    f"Summary of telemetry files (all times in UTC):\n\n"
    f"- metric.csv: {metric_rows} rows. Sample services: {', '.join(metric_cmdb_sample)}. "
    f"Sample KPI names: {', '.join(metric_name_sample)}. Earliest shown timestamp: {first_metric_ts}.\n\n"
    f"- trace.csv: {trace_rows} rows. No trace telemetry available.\n\n"
    f"- log.csv: {log_rows} rows. Sample services: {', '.join(log_cmdb_sample)}. "
    f"Sample log names: {', '.join(log_name_sample)}.\n\n"
    f"- error_logs.csv: {error_rows} rows. No raw error messages available.\n\n"
    f"Overall: metric and log telemetry are present and cover multiple Sock Shop services (e.g., carts, catalogue, front-end, orders, payment, user and their DBs). "
    f"Trace data and error log records are missing."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  rows                                    sample_cmdb_ids                                       sample_names
0  metric.csv  2000  [carts, carts-db, catalogue, catalogue-db, fro...  [cpu, error, latency-50, latency-90, mem, sock...,                   timestamp cmdb_id    kpi_name         value
0 2024-01-19 04:16:00+00:00   carts         cpu  1.667407e+00
1 2024-01-19 04:16:00+00:00   carts       error  0.000000e+00
2 2024-01-19 04:16:00+00:00   carts  latency-50  4.894504e-01
3 2024-01-19 04:16:00+00:00   carts  latency-90  2.024034e+00
4 2024-01-19 04:16:00+00:00   carts         mem  2.169774e+08,         file  rows sample_cmdb_ids sample_names
0  trace.csv     0              []           [], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [],       file  rows                                    sample_cmdb_ids                      sample_names
0  log.csv   450  [carts-db, catalogue, front-end, orders, payme...  [log.error_count, log.row_count],                   timestamp    cmdb_id         log_name  value
0 2024-01-19 04:16:00+00:00   carts-db  log.error_count      0
1 2024-01-19 04:16:00+00:00   carts-db    log.row_count      8
2 2024-01-19 04:16:00+00:00  catalogue  log.error_count      0
3 2024-01-19 04:16:00+00:00  catalogue    log.row_count     12
4 2024-01-19 04:16:00+00:00  front-end  log.error_count      0,              file  rows sample_cmdb_ids sample_names
0  error_logs.csv     0              []         None, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

# Aggregate global (full-series) stats by cmdb_id and kpi_name
agg_full = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False).value.agg(
    global_p95=lambda s: s.quantile(0.95),
    total_points='count'
).reset_index()

# Define incident window in UTC
window_start = pd.to_datetime("2024-01-19 04:13:55", utc=True)
window_end   = pd.to_datetime("2024-01-19 04:43:55", utc=True)

# Filter metric to the incident window
metric_window = metric[(metric['timestamp'] >= window_start) & (metric['timestamp'] <= window_end)].copy()

# Aggregate windowed stats by cmdb_id and kpi_name
agg_window = metric_window.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    window_count=('value', 'count'),
    window_max=('value', 'max'),
    window_mean=('value', 'mean'),
    window_earliest_timestamp=('timestamp', 'min'),
    window_latest_timestamp=('timestamp', 'max')
).reset_index()

# Merge full-series aggregates with windowed aggregates (keep all groups from full-series)
agg_all = pd.merge(agg_full, agg_window, on=['cmdb_id', 'kpi_name'], how='left')

# For groups with no points in the window, set window_count to 0 (leave other window stats as NaN/NaT)
agg_all['window_count'] = agg_all['window_count'].fillna(0).astype(int)

# Sort by window_max descending (NaNs will be placed last)
agg_all_sorted = agg_all.sort_values(by='window_max', ascending=False, na_position='last')

# Keep the full aggregated results in kernel variables for later use:
# - agg_full (global p95 and total points)
# - agg_window (window-specific aggregates for groups present in window)
# - agg_all (merged full + window aggregates)
# - agg_all_sorted (sorted merged results)
# Return only top 20 rows with the requested columns
top20 = agg_all_sorted.loc[:, [
    'cmdb_id', 'kpi_name', 'global_p95', 'total_points',
    'window_count', 'window_max', 'window_mean',
    'window_earliest_timestamp', 'window_latest_timestamp'
]].head(20)

top20

```
Out[2]:
```
```python
# Build a plain-English summary string from the previously computed top20 results
rows = top20.head(10)
lines = []
for i, r in rows.reset_index(drop=True).iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    win_max = int(round(r['window_max'])) if pd.notna(r['window_max']) else None
    gp95 = int(round(r['global_p95'])) if pd.notna(r['global_p95']) else None
    wc = int(r['window_count'])
    lines.append(f"{i+1}) {cmdb} ({kpi}) — window_max={win_max:,}  global_p95≈{gp95:,}  window_count={wc}")

earliest = rows['window_earliest_timestamp'].dropna().min()
latest = rows['window_latest_timestamp'].dropna().max()

summary = (
    "Incident window: 2024-01-19 04:13:55 to 2024-01-19 04:43:55 UTC.\n\n"
    "Top findings (sorted by window_max):\n"
    f"{chr(10).join(lines)}\n\n"
    f"Earliest window timestamp among top groups: {earliest}\n"
    f"Latest window timestamp among top groups:   {latest}\n\n"
    "Interpretation:\n"
    "- The dominant issue during the incident window is high memory usage across many services. "
    "The highest observed memory spike is on catalogue-db (≈432,513,600), followed by orders, shipping, queue-master, carts, and carts-db.\n"
    "- Disk I/O spikes also appear (notably on carts-db and user-db) but the top ranks are dominated by memory (mem) KPIs.\n"
    "- All top groups show window_count=25, indicating sustained abnormal values throughout the sampled window (regular per-minute samples present).\n\n"
    "Conclusion: memory usage spikes (catalogue-db highest) are the primary signal in the incident window; investigate memory pressure on catalogue-db and other memory-peaking services."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name    global_p95  total_points  window_count    window_max   window_mean window_earliest_timestamp   window_latest_timestamp
21       catalogue-db      mem  4.324630e+08            25            25  4.325136e+08  4.324380e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
34             orders      mem  3.312808e+08            25            25  3.313166e+08  3.312355e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
66           shipping      mem  3.066617e+08            25            25  3.067358e+08  3.066610e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
50       queue-master      mem  2.682980e+08            25            25  2.683003e+08  2.682915e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
4               carts      mem  2.192675e+08            25            25  2.193284e+08  2.183089e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
9            carts-db      mem  2.185122e+08            25            25  2.186452e+08  2.176670e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
78            user-db      mem  2.036762e+08            25            25  2.039300e+08  2.011709e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
39          orders-db      mem  1.139273e+08            25            25  1.140397e+08  1.130145e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
27          front-end      mem  1.061103e+08            25            25  1.063166e+08  1.032408e+08 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
54           rabbitmq      mem  9.845928e+07            25            25  9.848422e+07  9.845418e+07 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
16          catalogue      mem  1.368348e+07            25            25  1.475953e+07  9.179358e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
73               user      mem  8.745670e+06            25            25  8.774178e+06  8.734020e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
45            payment      mem  4.798327e+06            25            25  4.800512e+06  4.773510e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
57  rabbitmq-exporter      mem  4.575232e+06            25            25  4.575232e+06  4.575232e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
8            carts-db   diskio  3.478192e+06            25            25  3.953832e+06  3.174812e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
77            user-db   diskio  2.804097e+06            25            25  3.039788e+06  2.495967e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
61         session-db      mem  2.948970e+06            25            25  2.956356e+06  2.907587e+06 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
38          orders-db   diskio  8.967222e+05            25            25  9.131440e+05  8.028296e+05 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
49       queue-master   diskio  5.737322e+04            25            25  1.335557e+05  3.546223e+04 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00
60         session-db   diskio  1.813749e+04            25            25  1.868727e+04  3.483559e+03 2024-01-19 04:16:00+00:00 2024-01-19 04:40:00+00:00```
```

In [ ]:
# In[3]:

import pandas as pd
# Reuse previously loaded DataFrames: metric, log
# Ensure timestamp columns are parsed (they should already be, but enforce)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["carts", "catalogue", "orders", "payment", "user"]

# Ensure global aggregates (agg_full) exist; compute if not
if 'agg_full' not in globals():
    agg_full = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False).value.agg(
        global_p95=lambda s: s.quantile(0.95),
        total_points='count'
    ).reset_index()

# Define incident window (UTC)
window_start = pd.to_datetime("2024-01-19 04:13:55", utc=True)
window_end   = pd.to_datetime("2024-01-19 04:43:55", utc=True)

# Metric rows within window
metric_window = metric[(metric['timestamp'] >= window_start) & (metric['timestamp'] <= window_end)].copy()

# Compute window aggregates if not present
if 'agg_window' not in globals():
    agg_window = metric_window.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
        window_count=('value', 'count'),
        window_max=('value', 'max'),
        window_mean=('value', 'mean'),
        window_earliest_timestamp=('timestamp', 'min'),
        window_latest_timestamp=('timestamp', 'max')
    ).reset_index()

# Build candidate metric table: join global p95 with window stats, filter by candidates and window_count>0
metric_candidates_full = pd.merge(
    agg_full,
    agg_window,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Keep only candidate cmdb_ids and rows where window_count>0
metric_candidates = metric_candidates_full[metric_candidates_full['cmdb_id'].isin(candidates)].copy()

# Replace NaN window_count with 0 then filter >0
metric_candidates['window_count'] = metric_candidates['window_count'].fillna(0).astype(int)
metric_candidates = metric_candidates[metric_candidates['window_count'] > 0].copy()

# Compute severity_ratio = window_max / global_p95 (if global_p95>0 else NaN)
metric_candidates['severity_ratio'] = metric_candidates.apply(
    lambda r: (r['window_max'] / r['global_p95']) if pd.notna(r['global_p95']) and r['global_p95'] > 0 else float('nan'),
    axis=1
)

# Select and sort compactly, limit to 50 rows
metric_candidates_summary = metric_candidates.loc[:, [
    'cmdb_id', 'kpi_name', 'global_p95', 'window_max', 'window_count',
    'window_earliest_timestamp', 'severity_ratio'
]].sort_values(by='severity_ratio', ascending=False).head(50)

# ---- Part 2: log.csv aggregations for candidates within the incident window ----
log_window = log[(log['timestamp'] >= window_start) & (log['timestamp'] <= window_end)].copy()

# For each candidate, compute sums and counts for log.error_count and sample log_name values
def compute_log_aggregates(df_window, candidates_list):
    rows = []
    for c in candidates_list:
        df_c = df_window[df_window['cmdb_id'] == c]
        total_log_rows = len(df_c)
        # filter error_count entries
        err_df = df_c[df_c['log_name'] == 'log.error_count']
        sum_errors = err_df['value'].sum() if not err_df.empty else 0
        max_error = err_df['value'].max() if not err_df.empty else 0
        # earliest timestamp where value>0
        if not err_df.empty and (err_df['value'] > 0).any():
            earliest_error_ts = err_df[err_df['value'] > 0]['timestamp'].min()
        else:
            earliest_error_ts = pd.NaT
        sample_log_names = list(pd.unique(df_c['log_name'])[:10]) if not df_c.empty else []
        rows.append({
            'cmdb_id': c,
            'sum_errors': sum_errors,
            'max_error': max_error,
            'earliest_error_ts': earliest_error_ts,
            'total_log_rows_in_window': total_log_rows,
            'sample_log_names': sample_log_names
        })
    return pd.DataFrame(rows)

log_candidates_summary = compute_log_aggregates(log_window, candidates)

# Keep full intermediate results in kernel variables:
# metric_candidates_full, metric_candidates, metric_candidates_summary, log_window, log_candidates_summary

# Display compact summaries (as DataFrames)
metric_candidates_summary, log_candidates_summary

```
Out[3]:
```
Summary (using metric.csv and log.csv, window 2024-01-19 04:13:55–04:43:55 UTC):

What was computed
- For each candidate service (carts, catalogue, orders, payment, user) we computed global P95 from the full metric series and window stats (window_max, window_count, earliest timestamp) and a severity_ratio = window_max / global_p95.
- From logs we summed log.error_count and extracted max/earliest error timestamp and total log rows in the window.

Key metric findings (sorted by severity_ratio)
- carts — cpu: severity_ratio ≈ 1.28 (window_max ≈ 2.918 vs global_p95 ≈ 2.284), window_count = 25.
- carts — latency-90: severity_ratio ≈ 1.21, window_count = 25.
- orders — latency-50: severity_ratio ≈ 1.16, window_count = 25.
- orders — latency-90: severity_ratio ≈ 1.13, window_count = 25.
- catalogue — mem: severity_ratio ≈ 1.08 (window_max ≈ 14,759,530 vs global_p95 ≈ 13,683,480), window_count = 25.
- Many other KPI rows exist for these services; some KPI types (e.g., 'error') have global_p95 = 0, producing NaN severity_ratio.

Log findings (one row per candidate)
- carts: sum_errors = 0, max_error = 0, no earliest_error_ts, total_log_rows_in_window = 42, sample log names: [log.error_count, log.row_count].
- catalogue, orders, payment, user: sum_errors = 0, max_error = 0, no earliest_error_ts, total_log_rows_in_window = 50, sample log names: [log.error_count, log.row_count].

Interpretation and recommended focus
- No log.error_count entries >0 for any candidate service during the window (logs do not show explicit errors).
- Metric signals point to modest but sustained anomalies:
  - carts shows the strongest relative anomalies (CPU and tail latency).
  - orders shows elevated latency (both median/tail metrics).
  - catalogue shows elevated memory, but its severity_ratio is smaller than carts’ CPU/latency.
- Severity ratios are mostly in the ~1.08–1.28 range (not massive one-off spikes) and window_count = 25 for these KPIs, indicating sustained deviation across the sampled window.

Suggested next steps
- Investigate carts first (CPU and latency metrics), then check orders latency and catalogue memory pressure.
- Correlate these metric anomalies with application traces or deeper logs (if available) and check resource usage on the hosts/containers for these services.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name    global_p95    window_max  window_count window_earliest_timestamp  severity_ratio
0       carts         cpu  2.284256e+00  2.918080e+00            25 2024-01-19 04:16:00+00:00        1.277475
3       carts  latency-90  3.173758e+00  3.845343e+00            25 2024-01-19 04:16:00+00:00        1.211606
32     orders  latency-50  2.302544e-01  2.672860e-01            25 2024-01-19 04:16:00+00:00        1.160829
33     orders  latency-90  1.145196e+00  1.294782e+00            25 2024-01-19 04:16:00+00:00        1.130621
16  catalogue         mem  1.368348e+07  1.475953e+07            25 2024-01-19 04:16:00+00:00        1.078638
..        ...         ...           ...           ...           ...                       ...             ...
12  catalogue      diskio  0.000000e+00  0.000000e+00            25 2024-01-19 04:16:00+00:00             NaN
13  catalogue       error  0.000000e+00  0.000000e+00            25 2024-01-19 04:16:00+00:00             NaN
31     orders       error  0.000000e+00  0.000000e+00            25 2024-01-19 04:16:00+00:00             NaN
42    payment       error  0.000000e+00  0.000000e+00            25 2024-01-19 04:16:00+00:00             NaN
70       user       error  0.000000e+00  0.000000e+00            25 2024-01-19 04:16:00+00:00             NaN

[36 rows x 7 columns],      cmdb_id  sum_errors  max_error earliest_error_ts  total_log_rows_in_window                  sample_log_names
0      carts           0          0               NaT                        42  [log.error_count, log.row_count]
1  catalogue           0          0               NaT                        50  [log.error_count, log.row_count]
2     orders           0          0               NaT                        50  [log.error_count, log.row_count]
3    payment           0          0               NaT                        50  [log.error_count, log.row_count]
4       user           0          0               NaT                        50  [log.error_count, log.row_count])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```